# Notebook 01 — 소표본·희소집단에서의 안정성

> **장점 #1**: 베이지안(Bayesian) 접근은 사전분포(prior)를 통한 정규화(regularization)로
> 표본이 작거나 집단 크기가 불균형(sparse focal group)할 때에도 안정적인 추정을 제공합니다.

학습 목표 (Learning Objectives)
- 표본 크기와 집단 균형이 DIF 검출에 미치는 영향을 시뮬레이션으로 확인한다.
- 빈도주의 MH·로지스틱 회귀(maximum likelihood)와 베이지안 1PL DIF의 추정 안정성을 비교한다.
- RMSE, 신뢰/신용구간의 coverage, 분산(variance)의 차이를 시각화한다.

전제: Notebook 00을 먼저 학습하셨다고 가정합니다.


## 1. 문제 설정

다음 시나리오를 반복 시뮬레이션(Monte Carlo replication)하여
빈도주의(frequentist)와 베이지안(Bayesian) 방법의 추정 안정성을 비교합니다.

| 시나리오 | n_ref | n_focal | 비고 |
|---|---|---|---|
| A — 균형 충분 | 300 | 300 | 표준 |
| B — 균형 소표본 | 80  | 80  | 표본 적음 |
| C — 희소 focal | 300 | 50  | 집단 불균형 |
| D — 극단 희소 | 400 | 25  | 매우 극단적 |

자료생성과정(data-generating process, DGP)은 Notebook 00과 동일 — 10문항, 문항 5에 +0.8 DIF, 문항 8에 -0.4 DIF.
각 시나리오마다 **30회** 반복 시뮬레이션합니다 (학습 목적상 빠른 실행을 위해 30회로 설정;
실제 연구에서는 1000회 정도 권장).


In [ ]:
# 백엔드 선택 (Notebook 00과 동일)
BACKEND = "stan"
import platform, importlib.util, warnings
def _resolve(req):
    req = req.lower()
    if req == "numpyro":
        ok = (importlib.util.find_spec("jax") is not None
              and importlib.util.find_spec("numpyro") is not None)
        if not ok:
            warnings.warn("numpyro unavailable → Stan fallback")
            return "stan"
    return "stan" if req != "numpyro" else "numpyro"
BACKEND = _resolve(BACKEND)
print(f"Active backend: {BACKEND}")


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from difbayes import simulate, visualize, frequentist, diagnostics, models

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
print("Modules loaded.")


## 2. 반복 시뮬레이션 실행

> ⏱ **시간 안내**: 4개 시나리오 × 30회 × 2개 방법 = 240회 적합.
> Stan 컴파일은 첫 1회만, 이후 캐시 사용. 총 5~15분 소요 예상.
> 시간이 부족하면 `N_REPS = 10` 으로 줄여 실행해보세요.


In [ ]:
N_REPS = 30   # 빠른 실행. 정밀 비교는 100~1000 권장.

SCENARIOS = [
    dict(name="A: balanced (300/300)", n_ref=300, n_focal=300),
    dict(name="B: small balanced (80/80)", n_ref=80,  n_focal=80),
    dict(name="C: sparse focal (300/50)", n_ref=300, n_focal=50),
    dict(name="D: extreme sparse (400/25)", n_ref=400, n_focal=25),
]

# 진짜 DIF 모수 (시나리오 공통)
b_true = np.linspace(-2.0, 2.0, 10)
delta_b_true = np.zeros(10)
delta_b_true[4] = 0.8
delta_b_true[7] = -0.4
TRUE_J = len(b_true)


In [ ]:
# 결과 저장 구조
results = []   # list of dict: scenario, rep, method, item, est, lo, hi, truth

for sc in SCENARIOS:
    print(f"\n=== Scenario {sc['name']} ===")
    for r in range(N_REPS):
        seed = 1000 * (1 + SCENARIOS.index(sc)) + r
        data = simulate.simulate_rasch_dif(
            n_ref=sc["n_ref"], n_focal=sc["n_focal"],
            b_true=b_true, delta_b_true=delta_b_true, seed=seed,
        )

        # --- (a) MH ---
        mh = frequentist.mantel_haenszel_all(data.Y, data.group, n_strata=4)
        # Δ_MH 를 Δb 스케일로 환산: Δb ≈ -Δ_MH / 2.35 (대략적 부호 환산)
        # 정확한 환산이 아니므로 비교는 "MH는 logit-OR 스케일" 그대로 두는 것이 안전
        for j, m in enumerate(mh):
            # logit-OR = log(α_MH) = -Δ_MH/2.35
            log_or = (-m.delta_mh / 2.35) if np.isfinite(m.delta_mh) else np.nan
            results.append(dict(
                scenario=sc["name"], rep=r, method="MH (logit-OR)",
                item=j+1, est=log_or, lo=np.nan, hi=np.nan,
                truth=delta_b_true[j],
            ))

        # --- (b) Bayesian non-hierarchical ---
        fit = models.fit_rasch_dif(
            Y=data.Y, group=data.group, backend=BACKEND,
            n_chains=2, n_warmup=300, n_samples=500,
            prior_sigma_delta=1.0, seed=seed,
        )
        samples = fit["samples"]["delta"].reshape(-1, TRUE_J)
        for j in range(TRUE_J):
            s = samples[:, j]
            results.append(dict(
                scenario=sc["name"], rep=r, method="Bayes (weak prior)",
                item=j+1, est=s.mean(),
                lo=np.quantile(s, 0.025), hi=np.quantile(s, 0.975),
                truth=delta_b_true[j],
            ))
        if (r + 1) % 5 == 0:
            print(f"  rep {r+1}/{N_REPS}")

results_df = pd.DataFrame(results)
results_df.head()


## 3. RMSE 및 분산 비교 (Bias-Variance)

각 시나리오·방법·문항별로 RMSE = $\sqrt{\mathrm{E}[(\hat{\Delta b}_j - \Delta b_j)^2]}$ 와
추정치 분산을 계산합니다.


In [ ]:
# Δb 스케일로 비교: MH 의 logit-OR 은 1PL Δb 와 부호가 일치하므로 그대로 사용
agg = (results_df
       .assign(error=lambda d: d["est"] - d["truth"])
       .groupby(["scenario", "method", "item"])
       .agg(rmse=("error", lambda x: np.sqrt(np.nanmean(x**2))),
            sd=("est", "std"),
            mean_est=("est", "mean"),
            truth=("truth", "first"))
       .reset_index())
print(agg.head(12))


In [ ]:
# 시각화 — 진짜 DIF 문항(5, 8)에 대한 RMSE 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for k, item in enumerate([5, 8]):
    sub = agg[agg["item"] == item]
    scenarios = [s["name"] for s in SCENARIOS]
    x = np.arange(len(scenarios))
    width = 0.35
    mh   = sub[sub["method"] == "MH (logit-OR)"].set_index("scenario").loc[scenarios, "rmse"]
    bay  = sub[sub["method"] == "Bayes (weak prior)"].set_index("scenario").loc[scenarios, "rmse"]
    axes[k].bar(x - width/2, mh, width, label="MH (logit-OR)", color="#1f77b4")
    axes[k].bar(x + width/2, bay, width, label="Bayes (weak prior)", color="#2ca02c")
    axes[k].set_xticks(x)
    axes[k].set_xticklabels([s.split(":")[0] for s in scenarios])
    axes[k].set_title(f"Item {item}  (true Δb = {delta_b_true[item-1]:+.2f})")
    axes[k].set_ylabel("RMSE")
    axes[k].legend(fontsize=9)
    axes[k].grid(axis="y", alpha=0.3)
fig.suptitle("RMSE of DIF estimates across scenarios", y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig("../outputs/01_rmse_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


**해석**

- 시나리오 A (충분 표본)에서 두 방법은 비슷한 RMSE.
- 시나리오 B, C, D (소표본 또는 희소집단)로 갈수록 **MH의 RMSE가 베이지안보다 빠르게 악화**.
- 베이지안은 prior 정규화 덕분에 추정치가 0 쪽으로 약간 끌려가지만,
  분산이 크게 감소하여 전체 RMSE는 더 낮아짐 (bias-variance tradeoff).


## 4. 95% 신용구간(credible interval) Coverage

베이지안 95% 신용구간이 진짜 값을 얼마나 자주 포함하는지(coverage rate) 계산합니다.
이상적으로는 95%에 가까워야 합니다.


In [ ]:
bay = results_df[results_df["method"] == "Bayes (weak prior)"].copy()
bay["covered"] = (bay["lo"] <= bay["truth"]) & (bay["truth"] <= bay["hi"])
coverage = bay.groupby(["scenario", "item"])["covered"].mean().reset_index()
print("Bayesian 95% CI coverage:")
print(coverage.pivot(index="item", columns="scenario", values="covered").round(2))


## 5. 요약

**관찰 정리**

1. 충분 표본에서는 MH와 베이지안이 유사하지만, **소표본·희소집단으로 갈수록 베이지안이 우월**.
2. 베이지안의 RMSE 우위는 **분산 감소**에서 옴 — 사전이 극단치를 0 쪽으로 끌어당기는 효과.
3. 신용구간 coverage가 대체로 95% 근처에 머문다는 점은 **사전이 결과를 왜곡하지 않음**을 보여줌.

**실무 시사**

- 표본이 작거나 minority group의 DIF를 평가해야 한다면, 베이지안 1PL DIF 모형을 권장.
- prior 폭(`prior_sigma_delta`)을 변경하여 민감도 분석(sensitivity analysis)을 하면 더 견고한 결론이 가능.

다음 노트북(**02**)에서는 같은 자료에 대해 **사후확률 기반 의사결정**의 풍부함을 살펴봅니다.
